<a href="https://colab.research.google.com/github/zFonta/CEIA-TF-Chess-DL/blob/main/notebooks/03_train_resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Entorno de entrenamiento y linea base ResNet

Cubre las tareas **4.1** (entorno de entrenamiento) y **4.2** (arquitectura
residual) del WBS. El alcance completo del bloque esta en
[`docs/modelado.md`](../docs/modelado.md).

Lo que hace esta notebook, en orden:

1. Baja el dataset etiquetado desde Hugging Face.
2. Parte los datos **por partida**, no por posicion.
3. Precomputa el cache de tensores.
4. Mide los dos baselines no neuronales, que fijan el piso de RMSE.
5. Construye la ResNet y verifica que aprende.

> **Runtime: GPU (T4).** Al reves que las notebooks del pipeline de datos, que
> pedian CPU. Un modelo de ~3 M de parametros sobre entradas de 8x8 no
> justifica una A100: quema unidades ~6 veces mas rapido y queda
> desaprovechada.


## 1. Entorno


In [1]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija. Entrenar no lo usa, pero sin el se saltean
# los 17 tests de integracion del pipeline, que son la evidencia del
# requerimiento 3.2 para la memoria.
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())



Listo. Directorio de trabajo: /content/CEIA-TF-Chess-DL


In [2]:
import torch
from chessdl.colab import TRAINING, describe_runtime

runtime = describe_runtime(phase=TRAINING)
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Modelo de GPU  :", torch.cuda.get_device_name(0))
print("CPU workers    :", runtime.cpu_count)
print("torch          :", torch.__version__)
for aviso in runtime.warnings():
    print("AVISO:", aviso)

GPU disponible : True
Modelo de GPU  : Tesla T4
CPU workers    : 2
torch          : 2.11.0+cu128


## 2. Tests

Conviene correrlos antes de gastar cuota de GPU. La salida sirve ademas como
evidencia de los requerimientos de testing (3.1 y 3.2) para la memoria.


In [3]:
!{sys.executable} -m pytest -q

........................................................................ [ 25%]
........................................................................ [ 51%]
........................................................................ [ 77%]
................................................................         [100%]
280 passed in 52.11s


## 3. Cargar el dataset


In [4]:
from chessdl.config import load_config
from chessdl import hf
from chessdl.data import schema

cfg = load_config()
directorio = hf.download_dataset(cfg.output.hf_repo_id, "/content/ceia-chess/hub")
tabla = schema.read_dataset(schema.shard_paths(directorio))
print(f"{tabla.num_rows:,} posiciones")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 158 files:   0%|          | 0/158 [00:00<?, ?it/s]

2,552,804 posiciones


In [5]:
import numpy as np

# Solo las columnas que necesita el entrenamiento: el resto no entra en RAM.
fens     = tabla["fen"].to_pylist()
game_ids = tabla["game_id"].to_pylist()
# El target es value_stm: la etiqueta en perspectiva del jugador al turno,
# que es la que corresponde a la entrada espejada.
targets  = np.asarray(tabla["value_stm"], dtype=np.float32)
print(f"{len(fens):,} posiciones de {len(set(game_ids)):,} partidas")

2,552,804 posiciones de 772,797 partidas


## 4. Particion por partida

**La decision mas importante del bloque.** Hay 4 posiciones por partida, y
las de una misma partida comparten apertura, jugadores y estructura. Partir
por posicion deja hermanas a ambos lados del muro: la validacion queda
inflada y el error recien aparece cuando el motor juega peor que lo que
prometian las metricas.

La asignacion es por *hash* del `game_id`, no por mezcla de la lista. Asi es
**estable si el dataset crece**: agregar shards mañana deja cada partida en
el split que ya tenia.


In [6]:
from chessdl.training.split import describe_split, leaked_games, split_masks

split_cfg = cfg.training.split_config()
masks = split_masks(game_ids, split_cfg)

print(describe_split(masks, game_ids))
print()
fugadas = leaked_games(masks, game_ids)
print("Partidas en mas de un split:", len(fugadas), "(tiene que ser 0)")
assert not fugadas

split       posiciones       %    partidas
------------------------------------------
train        2,297,785  90.01%     695,480
val            127,520   5.00%      38,654
test           127,499   4.99%      38,663

Partidas en mas de un split: 0 (tiene que ser 0)


## 5. Cache de tensores

Codificar un FEN cuesta ~79 microsegundos, de los cuales 44 son parsear el
FEN. Con 2 vCPU son ~100 segundos por epoca solo codificando, contra ~140
que tarda la T4: sin cache el tiempo por epoca casi se duplica y la mitad de
las unidades se va en esperar a la CPU.

El cache son 2,9 GB en `uint8` y se construye una sola vez. **No modifica el
dataset**: el Parquet sigue guardando FEN.


In [7]:
from chessdl.training.cache import build_cache, cache_path_for, cache_size_bytes, load_cache

ruta_cache = cache_path_for(cfg.training.cache_dir)
print(f"Tamano estimado: {cache_size_bytes(len(fens))/1e9:.1f} GB -> {ruta_cache}")

if not ruta_cache.exists():
    build_cache(fens, ruta_cache, progress=True)

cache = load_cache(ruta_cache, expected_rows=len(fens))
print("Cache listo:", cache.shape, cache.dtype)

Tamano estimado: 2.9 GB -> /content/ceia-chess/cache/tensors.npy


encoding:   0%|          | 0/2552804 [00:00<?, ?pos/s]

Cache listo: (2552804, 18, 8, 8) uint8


In [8]:
# El cache tiene que ser indistinguible del encoder: si no, el modelo se
# entrena con una representacion y el motor corre con otra, sin que falle nada.
import chess
from chessdl.encoding import board_to_tensor
from chessdl.training.cache import cached_to_tensor

rng = np.random.default_rng(0)
iguales = 0
for i in rng.choice(len(fens), size=500, replace=False):
    directo = board_to_tensor(chess.Board(fens[i]))
    iguales += np.array_equal(cached_to_tensor(np.asarray(cache[i])), directo)
print(f"Posiciones verificadas: 500   identicas al encoder: {iguales}")
assert iguales == 500

Posiciones verificadas: 500   identicas al encoder: 500


## 6. Baselines no neuronales

Un RMSE suelto no se puede interpretar. Estos dos lo acotan:

- **Media constante** — el piso absoluto. Un modelo que no le gane tiene un
  error de implementacion, no un problema de arquitectura.
- **Material lineal** — el piso "sabe algo de ajedrez". La diferencia entre
  este y la red es lo que la red realmente aporta.


In [9]:
from chessdl.training.baselines import describe_weights, material_baseline, mean_baseline

idx_train = np.flatnonzero(masks['train'])
idx_val   = np.flatnonzero(masks['val'])

# Submuestra para el ajuste lineal: con 2,3 M de filas no hace falta mas.
sub_train = rng.choice(idx_train, size=min(200_000, len(idx_train)), replace=False)

media = mean_baseline(targets[idx_train], targets[idx_val])
material, pesos = material_baseline(
    np.asarray(cache[np.sort(sub_train)]), targets[np.sort(sub_train)],
    np.asarray(cache[idx_val]), targets[idx_val],
)
print(media)
print(material)

media constante             RMSE 0.4886   MAE 0.3778
material (lineal)           RMSE 0.3973   MAE 0.3053


In [10]:
# Lo que tiene que cumplirse es el ORDEN de las piezas, rey = 0, y un sesgo
# cercano al valor de tener la jugada. Las razones NO se comparan con
# 1/3/3/5/9: esa es la escala en centipeones y el ajuste es en espacio
# value, comprimido por tanh. La propia tabla lo aclara abajo.
print(describe_weights(pesos))

pieza           peso   en peones
--------------------------------
peon          0.1737        1.00
caballo       0.3134        1.80
alfil         0.3644        2.10
torre         0.5157        2.97
dama          0.8440        4.86
rey           0.0000        0.00
sesgo         0.0820

Las razones NO deben compararse con 1/3/3/5/9: esa es la escala en
centipeones. El ajuste es en espacio value, comprimido por tanh, donde
una dama (+900 cp -> 0,978) vale ~4 peones (+100 cp -> 0,245) y no 9.
Lo que tiene que cumplirse es el orden, rey = 0, y un sesgo cercano al
valor de tener la jugada.


## 7. La ResNet (tarea 4.2)

Implementada desde cero. Una ResNet de `torchvision` arranca con convolucion
7x7 de stride 2 y max-pooling: dejaria el tablero de 8x8 en 2x2 en dos pasos.
Aca la convolucion es 3x3 con padding 1, que mantiene 8x8 de punta a punta.

La salida pasa por `tanh`, asi el rango del **requerimiento 1.4** queda
garantizado por construccion y no por lo que la red haya aprendido.


In [11]:
from chessdl.models.resnet import ChessResNet, ResNetConfig

modelo = ChessResNet(ResNetConfig(channels=128, blocks=8))
print(modelo.describe())
print()
print(modelo)

ResNet 128 canales x 8 bloques (cabeza 32/256) -- 2,913,345 parametros

ChessResNet(
  (stem): Sequential(
    (0): Conv2d(18, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (blocks): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (norm1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (1): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (norm1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [12]:
# Dos comprobaciones distintas, que conviene no confundir.
modelo.eval()

# 1) La cota del requerimiento 1.4. El tanh la garantiza pase lo que pase.
with torch.no_grad():
    extremo = modelo(torch.randn(256, 18, 8, 8) * 100)
print(f"Entradas absurdas -> [{extremo.min():+.4f}, {extremo.max():+.4f}]   (requerimiento 1.4: [-1, 1])")
print("  Saturan, y esta bien: es ruido, no posiciones. Aca solo se verifica la cota.")

# 2) Que el tablero llegue a la cabeza. Lo que importa es que la salida VARIE
#    entre posiciones; si fuera identica para todas, la cabeza estaria
#    ignorando la entrada y eso no se ve en el chequeo de arriba.
#
#    La magnitud va a ser chica y eso es normal en una red sin entrenar: en
#    eval() BatchNorm normaliza con sus estadisticas iniciales (media 0,
#    varianza 1), que no son las de los datos. En train(), con las del lote, la
#    dispersion es un orden de magnitud mayor. No hay que leerlo como un problema.
muestra = np.sort(rng.choice(idx_val, size=min(1024, len(idx_val)), replace=False))
xr = torch.from_numpy(cached_to_tensor(np.asarray(cache[muestra])))
with torch.no_grad():
    reales = modelo(xr)
print(f"\nPosiciones reales -> desvio {reales.std():.5f}   rango [{reales.min():+.4f}, {reales.max():+.4f}]")
print("  Lo unico que se exige es desvio > 0: la prediccion depende del tablero.")
assert reales.std() > 0, "salida constante: la cabeza no ve el tablero"

with torch.no_grad():
    cuerpo = modelo.blocks(modelo.stem(torch.zeros(2, 18, 8, 8)))
print("\nEl tablero sigue siendo 8x8 tras el cuerpo:", tuple(cuerpo.shape))

Entradas absurdas -> [-1.0000, +1.0000]   (requerimiento 1.4: [-1, 1])
  Saturan, y esta bien: es ruido, no posiciones. Aca solo se verifica la cota.

Posiciones reales -> desvio 0.00613   rango [-0.0229, +0.0125]
  Lo unico que se exige es desvio > 0: la prediccion depende del tablero.

El tablero sigue siendo 8x8 tras el cuerpo: (2, 128, 8, 8)


### Que la red puede aprender

Antes de lanzar una campana completa, la prueba estandar: sobreajustar un
lote chico. Si no puede llevar la perdida a cero sobre 512 posiciones fijas,
el problema es estructural y ningun ajuste de hiperparametros lo arregla.


In [13]:
from torch.utils.data import DataLoader
from chessdl.training.dataset import PositionDataset

torch.manual_seed(0)
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'

lote_idx = np.sort(rng.choice(idx_train, size=512, replace=False))
mini = PositionDataset(targets, cache=cache, indices=lote_idx)
xb = torch.stack([mini[i][0] for i in range(len(mini))]).to(dispositivo)
yb = torch.tensor(mini.targets).to(dispositivo)

prueba = ChessResNet(ResNetConfig(channels=64, blocks=4)).to(dispositivo).train()
opt = torch.optim.Adam(prueba.parameters(), lr=3e-4)
for paso in range(300):
    opt.zero_grad()
    perdida = torch.nn.functional.mse_loss(prueba(xb), yb)
    perdida.backward(); opt.step()
    if paso % 50 == 0:
        print(f"  paso {paso:3d}: {perdida.item():.6f}")
print(f"\nPerdida final: {perdida.item():.6f}   (varianza de las etiquetas: {yb.var().item():.4f})")

  paso   0: 0.291251
  paso  50: 0.001592
  paso 100: 0.000036
  paso 150: 0.000014
  paso 200: 0.000032
  paso 250: 0.000008

Perdida final: 0.000006   (varianza de las etiquetas: 0.2307)


## 8. Resumen

Lo que queda listo para la primera campana de entrenamiento (tarea 4.4).


In [14]:
print(f"{'Posiciones':<34}{tabla.num_rows:>16,}")
print(f"{'Particion (train/val/test)':<34}{str(split_cfg.train)+'/'+str(split_cfg.val)+'/'+str(split_cfg.test):>16}")
print(f"{'Semilla de particion':<34}{split_cfg.seed:>16,}")
print(f"{'Cache de tensores':<34}{str(cache.shape):>16}")
print(f"{'Parametros de la ResNet':<34}{modelo.count_parameters():>16,}")
print()

# R2 respecto de predecir la media: hace interpretable el RMSE.
r2_material = 1 - (material.rmse / media.rmse) ** 2
print(f"{'':<34}{'RMSE':>10}{'MAE':>10}{'R2':>10}")
print("-" * 64)
print(f"{'media constante (piso absoluto)':<34}{media.rmse:>10.4f}{media.mae:>10.4f}{0.0:>10.3f}")
print(f"{'material lineal':<34}{material.rmse:>10.4f}{material.mae:>10.4f}{r2_material:>10.3f}")
print()
print(f"El material explica el {r2_material:.1%} de la varianza de la etiqueta.")
print(f"El {1-r2_material:.1%} restante -- posicion, seguridad del rey, peones pasados,")
print("tactica -- es lo que la red tiene que capturar para justificarse.")
print()
print(f"La red tiene que quedar por debajo de RMSE {material.rmse:.4f}.")

Posiciones                               2,552,804
Particion (train/val/test)           0.9/0.05/0.05
Semilla de particion                    20,260,911
Cache de tensores                 (2552804, 18, 8, 8)
Parametros de la ResNet                  2,913,345

                                        RMSE       MAE        R2
----------------------------------------------------------------
media constante (piso absoluto)       0.4886    0.3778     0.000
material lineal                       0.3973    0.3053     0.339

El material explica el 33.9% de la varianza de la etiqueta.
El 66.1% restante -- posicion, seguridad del rey, peones pasados,
tactica -- es lo que la red tiene que capturar para justificarse.

La red tiene que quedar por debajo de RMSE 0.3973.
